# DATA CLEANING - ABS Income (Data by Region) by SA2

This notebook cleans the ABS **Income (Data by Region)** dataset at the **SA2** level, using PySpark.

## Step 1 - Start the Spark session


In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("Data Cleaning - ABS Income")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/18 12:11:40 WARN Utils: Your hostname, Furina, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/18 12:11:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/whatcomesafter11/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/18 12:11:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Step 2 - Point to file
Spark can't read `.xlsx` directly, so we use `pandas` + `openpyxl` just to open the file and inspect it. 
If `openpyxl` isn't installed, run `pip install openpyxl` in a terminal first.

In [2]:
import pandas as pd

FILE_PATH = "../rea_data/landing/14100DO0004_2011-25.xlsx"

xls = pd.ExcelFile(FILE_PATH)
print("Sheets found in this workbook:")
for name in xls.sheet_names:
    print(" -", name)

Sheets found in this workbook:
 - Contents
 - Table 1
 - Table 2
 - Table 3


## Step 3 - Locate the table we use
`Table 1` from Step 2's output actually holds the income data (ABS names vary by release, so check the printed list above).

This prints the first 15 rows with **no header assumed**, so you can see exactly which row number the real column headers (e.g. `SA2 code`, `SA2 name`, `Median income ($)`) start on.

In [3]:
SHEET_NAME = "Table 1" 

preview = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, header=None, nrows=15)
preview

,0,1,2,3,4,5,6,7,8,9,...,46,47,48,49,50,51,52,53,54,55
0,Australian Bureau of Statistics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Data by region, 2011-25",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Released at 11.30am (Canberra time) 26 May 2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Table 1 INCOME (INCLUDING GOVERNMENT ALLOWANCE...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,Total personal income (weekly) - Persons aged ...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Gross capital gains - year ended 30 June,NaN,NaN,NaN
6,Code,Label,Year,$1-$499 per week (%),$500-$999 per week (%),$1000-$1999 per week (%),$2000-$2999 per week (%),$3000 or more per week (%),Nil income (%),Negative income (%),...,Superannuation and annuity income as main sour...,Total income earners (excl. Government pension...,Total income earners (excl. Government pension...,Total income (excl. Government pensions and al...,Median total income (excl. Government pensions...,Mean total income (excl. Government pensions a...,Persons who reported gross capital gains (no.),Total value of gross capital gains ($m),Median value of gross capital gains ($),Mean value of gross capital gains ($)
7,AUS,Australia,2011,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
8,AUS,Australia,2016,27.9,23.1,22.3,5.1,3.1,9,0.5,...,-,-,-,-,-,-,-,-,-,-
9,AUS,Australia,2017,-,-,-,-,-,-,-,...,-,-,-,-,-,-,955098,50532.4,3654,52908


## Step 4 - Load the sheet properly
Looking above, we find the row where the real column names appear. Everything above that row is title/notes junk we don't want.

This loads the full sheet with pandas using that header row, then hands it to Spark. 
We load everything as text (`str`) at this stage on purpose: it lets us differentiate between a genuinely blank value and a parsing mistake.

In [4]:
HEADER_ROW = 6

pdf_raw = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME, header=HEADER_ROW)

# Convert to Spark, everything as text for now
df_landing = spark.createDataFrame(pdf_raw.astype(str))
df_landing.printSchema()
df_landing.show(5, truncate=False)

/home/whatcomesafter11/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/whatcomesafter11/.venv/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- Code: string (nullable = true)
 |-- Label: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- $1-$499 per week (%): string (nullable = true)
 |-- $500-$999 per week (%): string (nullable = true)
 |-- $1000-$1999 per week (%): string (nullable = true)
 |-- $2000-$2999 per week (%): string (nullable = true)
 |-- $3000 or more per week (%): string (nullable = true)
 |-- Nil income (%): string (nullable = true)
 |-- Negative income (%): string (nullable = true)
 |-- Income inadequately described or not stated (%): string (nullable = true)
 |-- Median equivalised total household income (weekly) ($): string (nullable = true)
 |-- $1-$499 per week (%).1: string (nullable = true)
 |-- $500-$999 per week (%).1: string (nullable = true)
 |-- $1000-$1999 per week (%).1: string (nullable = true)
 |-- $2000-$2999 per week (%).1: string (nullable = true)
 |-- $3000 or more per week (%).1: string (nullable = true)
 |-- Nil income (%).1: string (nullable = true)
 |-- Partial i

## Step 5 - Clean up column names
ABS column headers are usually long and messy. This turns them into clean & code-friendly names.

After this cell, print the new column names.

In [5]:
import re

def clean_col_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^\w\s]", "", name)   # drop punctuation like $ ( ) %
    name = re.sub(r"\s+", "_", name)      # spaces -> underscores
    return name

new_names = [clean_col_name(c) for c in df_landing.columns]
df_landing = df_landing.toDF(*new_names)
print(df_landing.columns)

['code', 'label', 'year', '1499_per_week_', '500999_per_week_', '10001999_per_week_', '20002999_per_week_', '3000_or_more_per_week_', 'nil_income_', 'negative_income_', 'income_inadequately_described_or_not_stated_', 'median_equivalised_total_household_income_weekly_', '1499_per_week_1', '500999_per_week_1', '10001999_per_week_1', '20002999_per_week_1', '3000_or_more_per_week_1', 'nil_income_1', 'partial_income_stated_', 'all_incomes_not_stated_', 'age_pension_department_of_veterans_affairs_no', 'service_pension_department_of_veterans_affairs_no', 'income_support_supplement_department_of_veterans_affairs_no', 'employee_income_earners_no', 'employee_income_earners_median_age_years', 'total_employee_income_m', 'median_employee_income_', 'mean_employee_income_', 'employee_income_as_main_source_of_income_', 'own_unincorporated_business_income_earners_no', 'own_unincorporated_business_income_earners_median_age_years', 'total_own_unincorporated_business_income_m', 'median_own_unincorporated_

## Step 6 - Check for empty cells
Count how many missing values are in each column, so we know what we're dealing with before deciding how to handle them.

One ABS-specific thing to watch for: ABS often uses text placeholders like `"np"` (not published), `"-"`, or `"n.a."` instead of a blank cell. We deal with them in Step 7.

In [6]:
from functools import reduce

def null_check(df):
    has_null_condition = reduce(
        lambda acc, c: acc | F.col(c).isNull(),
        df.columns,
        F.lit(False)
    )

    summary = df.agg(
        F.count(F.lit(1)).alias("total_rows"),
        F.sum(has_null_condition.cast("int")).alias("rows_with_at_least_one_null"),
        *[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}") for c in df.columns]
    ).collect()[0]

    total = summary["total_rows"]
    rows_with_at_least_one_null = summary["rows_with_at_least_one_null"]

    print(f"Total rows: {total:,}")
    print(f"Rows with at least one null value: {rows_with_at_least_one_null:,}")
    print(f"Percentage of rows with at least one NULL: {round(100 * rows_with_at_least_one_null / total, 2)}%")
    print()
    for c in df.columns:
        print(f"{c:50s} {summary[f'null_{c}']:>12,}")

null_check(df_landing)

Total rows: 32,006
Rows with at least one null value: 7
Percentage of rows with at least one NULL: 0.02%

code                                                          4
label                                                         6
year                                                          7
1499_per_week_                                                7
500999_per_week_                                              7
10001999_per_week_                                            7
20002999_per_week_                                            7
3000_or_more_per_week_                                        7
nil_income_                                                   7
negative_income_                                              7
income_inadequately_described_or_not_stated_                  7
median_equivalised_total_household_income_weekly_             7
1499_per_week_1                                               7
500999_per_week_1                                             

## Step 7 - Remove Useless Columns


In [7]:
# idk

## Step 8 - Drop the years that we don't want

We only care about 2021 onwards.


In [9]:
before = df_landing.count()

df_landing = df_landing.filter(F.col("year").cast("double") >= 2021)

after = df_landing.count()
print(f"{before:,} rows -> {after:,} rows ({before - after:,} pre-2021 rows dropped)")
df_landing.select("year").distinct().orderBy("year").show()

32,006 rows -> 14,545 rows (17,461 pre-2021 rows dropped)
+------+
|  year|
+------+
|2021.0|
|2022.0|
|2023.0|
|2024.0|
|2025.0|
+------+



## Step 9 - Turn ABS placeholder text into real nulls, then cast types
This replaces known ABS "missing data" placeholders (`np`, `n.a.`, `-`, empty string) with a real `NULL`, then converts strings into numbers.

## ???
**You'll likely need to edit `NUMERIC_COLUMNS`** below to match the actual income columns you saw after Step 5 (e.g. `median_total_income`, `mean_total_income`, `median_age_of_earners`). Keep the SA2 code and name columns as text — don't add them to this list.

In [10]:
ABS_NULL_PLACEHOLDERS = ["np", "n.a.", "-", "", "nan", "none"]

NUMERIC_COLUMNS = [
    # "median_employee_income",
    # "median_total_income",
    # "mean_total_income",
]

df_clean = df_landing

# Step 7a: blank out ABS's placeholder strings across every column
for c in df_clean.columns:
    df_clean = df_clean.withColumn(
        c,
        F.when(F.lower(F.trim(F.col(c))).isin(ABS_NULL_PLACEHOLDERS), None)
         .otherwise(F.col(c))
    )

# Step 7b: strip $ and , out of numeric-looking columns, then cast to double
for c in NUMERIC_COLUMNS:
    df_clean = df_clean.withColumn(
        c,
        F.regexp_replace(F.col(c), r"[$,]", "").cast("double")
    )

df_clean.printSchema()

root
 |-- code: string (nullable = true)
 |-- label: string (nullable = true)
 |-- year: string (nullable = true)
 |-- 1499_per_week_: string (nullable = true)
 |-- 500999_per_week_: string (nullable = true)
 |-- 10001999_per_week_: string (nullable = true)
 |-- 20002999_per_week_: string (nullable = true)
 |-- 3000_or_more_per_week_: string (nullable = true)
 |-- nil_income_: string (nullable = true)
 |-- negative_income_: string (nullable = true)
 |-- income_inadequately_described_or_not_stated_: string (nullable = true)
 |-- median_equivalised_total_household_income_weekly_: string (nullable = true)
 |-- 1499_per_week_1: string (nullable = true)
 |-- 500999_per_week_1: string (nullable = true)
 |-- 10001999_per_week_1: string (nullable = true)
 |-- 20002999_per_week_1: string (nullable = true)
 |-- 3000_or_more_per_week_1: string (nullable = true)
 |-- nil_income_1: string (nullable = true)
 |-- partial_income_stated_: string (nullable = true)
 |-- all_incomes_not_stated_: string (n

## Step 10 - Identify the SA2 code and name columns
This checks that SA2 codes are the expected 9-digit ABS format and drops any row where the SA2 code is invalid.

In [11]:
SA2_CODE_COL = "code"  
SA2_NAME_COL = "label"  

before = df_clean.count()

df_clean = df_clean.filter(
    F.col(SA2_CODE_COL).isNotNull() &
    F.col(SA2_CODE_COL).rlike(r"^\d{9}$")
)

after = df_clean.count()
print(f"Dropped {before - after:,} rows with a missing/invalid SA2 code ({before:,} -> {after:,})")

Dropped 2,275 rows with a missing/invalid SA2 code (14,545 -> 12,270)


## Step 11 - Check for duplicate SA2 rows
Each SA2 region should appear once. This checks for duplicates on the SA2 code column and shows any offending rows.

In [12]:
dupes = (
    df_clean.groupBy(SA2_CODE_COL)
    .count()
    .filter(F.col("count") > 1)
)

print(f"SA2 codes appearing more than once: {dupes.count():,}")
dupes.show(20, truncate=False)

SA2 codes appearing more than once: 2,454
+---------+-----+
|code     |count|
+---------+-----+
|101041026|5    |
|102021055|5    |
|103041079|5    |
|105011093|5    |
|106031119|5    |
|105021098|5    |
|102011042|5    |
|104021085|5    |
|106011112|5    |
|101061542|5    |
|101021008|5    |
|103011060|5    |
|107011545|5    |
|101041022|5    |
|104021083|5    |
|104021088|5    |
|105011095|5    |
|102021045|5    |
|106021114|5    |
|107011132|5    |
+---------+-----+
only showing top 20 rows


## Step 12 - Save the cleaned data
Write to Parquet, then reload it and confirm the schema round-tripped correctly.

In [14]:
output_path = "../rea_data/curated/abs_income_sa2_clean.parquet"

df_clean.write.mode("overwrite").parquet(output_path)

df_verify = spark.read.parquet(output_path)
df_verify.printSchema()

assert df_verify.schema == df_clean.schema, "Schema mismatch after round-trip!"
print(f"Saved {df_verify.count():,} rows to {output_path} with schema intact.")

root
 |-- code: string (nullable = true)
 |-- label: string (nullable = true)
 |-- year: string (nullable = true)
 |-- 1499_per_week_: string (nullable = true)
 |-- 500999_per_week_: string (nullable = true)
 |-- 10001999_per_week_: string (nullable = true)
 |-- 20002999_per_week_: string (nullable = true)
 |-- 3000_or_more_per_week_: string (nullable = true)
 |-- nil_income_: string (nullable = true)
 |-- negative_income_: string (nullable = true)
 |-- income_inadequately_described_or_not_stated_: string (nullable = true)
 |-- median_equivalised_total_household_income_weekly_: string (nullable = true)
 |-- 1499_per_week_1: string (nullable = true)
 |-- 500999_per_week_1: string (nullable = true)
 |-- 10001999_per_week_1: string (nullable = true)
 |-- 20002999_per_week_1: string (nullable = true)
 |-- 3000_or_more_per_week_1: string (nullable = true)
 |-- nil_income_1: string (nullable = true)
 |-- partial_income_stated_: string (nullable = true)
 |-- all_incomes_not_stated_: string (n